# Notebook 03：因子构造

本 Notebook 使用月度研究面板和日频行情计算 README 中定义的价值、质量、动量、低波动、流动性和市值因子。原始因子按月末 `(date, stock_code)` 对齐；随后在每个调仓日进行 1%/99% 去极值、方向统一和横截面 Z-score 标准化。

## 1. 环境与路径

In [1]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.factors import (
    CORE_FACTOR_COLUMNS,
    FACTOR_PANEL_CONTEXT_COLUMNS,
    OPTIONAL_FACTOR_COLUMNS,
    build_raw_factor_panel,
    build_zscore_factor_panel,
    save_factor_panels,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MONTHLY_PANEL_PATH = PROCESSED_DIR / "monthly_panel.parquet"
CLEANED_PRICE_DAILY_PATH = PROCESSED_DIR / "price_daily_clean.parquet"
FACTOR_COLUMNS = (*CORE_FACTOR_COLUMNS, *OPTIONAL_FACTOR_COLUMNS)
OUTPUT_COLUMNS = (*FACTOR_PANEL_CONTEXT_COLUMNS, *FACTOR_COLUMNS)

print(f"项目目录：{PROJECT_ROOT}")
print(f"保留的上下文字段：{FACTOR_PANEL_CONTEXT_COLUMNS}")
print(f"因子列：{FACTOR_COLUMNS}")

项目目录：/Users/mac/Desktop/a-share-multifactor
保留的上下文字段：('date', 'stock_code', 'future_return_1m', 'label_available', 'label_status', 'is_in_label_sample', 'is_eligible', 'is_buyable', 'is_suspended', 'is_gross_profit_applicable', 'has_gp_factor_data', 'industry', 'market_cap')
因子列：('ep', 'bp', 'roe', 'gross_profitability', 'momentum_12_1', 'momentum_3m', 'low_volatility_60d', 'amihud_illiquidity_20d', 'size')


## 2. 读取月度面板与日频行情

财务类因子直接使用月度面板。动量、低波动和 Amihud 因子读取 Notebook 02 保存的清洗后日频面板；这里只读取计算所需字段，不再重复清洗主键、数值或复权价格。

In [2]:
monthly_panel = pd.read_parquet(MONTHLY_PANEL_PATH)
cleaned_price_daily = pd.read_parquet(
    CLEANED_PRICE_DAILY_PATH,
    columns=["date", "stock_code", "adjusted_close", "amount"],
)

print(
    f"月度面板：{monthly_panel.shape}，"
    f"区间 {monthly_panel['date'].min():%Y-%m-%d} 至 {monthly_panel['date'].max():%Y-%m-%d}"
)
print(
    f"清洗后日频行情：{cleaned_price_daily.shape}，"
    f"区间 {cleaned_price_daily['date'].min():%Y-%m-%d} 至 {cleaned_price_daily['date'].max():%Y-%m-%d}"
)

月度面板：(426474, 77)，区间 2018-01-31 至 2025-12-31
清洗后日频行情：(9286960, 4)，区间 2017-01-03 至 2025-12-31


## 3. 构造原始因子面板

行情因子的回看期按全市场交易日计算，并使用复权收盘价。精确回看日缺少行情时保留缺失值，不跨越停牌期取股票自身的上一条记录。输出面板只保留主键、标签、股票池状态、行业/市值控制变量和因子，不重复保存行情、财务原始值及中间清洗字段。

In [3]:
factor_panel_raw = build_raw_factor_panel(
    monthly_panel,
    cleaned_price_daily,
    include_optional=True,
)

assert len(factor_panel_raw) == len(monthly_panel)
assert not factor_panel_raw.duplicated(["date", "stock_code"]).any()
assert tuple(factor_panel_raw.columns) == OUTPUT_COLUMNS

del cleaned_price_daily
gc.collect()

raw_quality = pd.DataFrame({
    "非缺失数量": factor_panel_raw[list(FACTOR_COLUMNS)].notna().sum(),
    "缺失率": factor_panel_raw[list(FACTOR_COLUMNS)].isna().mean(),
    "无穷值数量": np.isinf(factor_panel_raw[list(FACTOR_COLUMNS)]).sum(),
})
raw_quality

,非缺失数量,缺失率,无穷值数量
ep,411917,0.034133,0
bp,422396,0.009562,0
roe,357674,0.161323,0
gross_profitability,402927,0.055213,0
momentum_12_1,392449,0.079782,0
momentum_3m,414561,0.027934,0
low_volatility_60d,404973,0.050416,0
amihud_illiquidity_20d,416710,0.022895,0
size,423069,0.007984,0


## 4. 横截面去极值与标准化

每个调仓日独立执行 1%/99% Winsorization 和 Z-score。LOWVOL 在原始定义中已经取负号；Amihud 在标准化时反向，使标准化因子统一为数值越大越好。Size 作为控制变量保留原方向。

In [4]:
factor_panel_zscore = build_zscore_factor_panel(factor_panel_raw)

zscore_summary = factor_panel_zscore.groupby("date")[list(FACTOR_COLUMNS)].agg(
    ["mean", lambda values: values.std(ddof=0)]
)
zscore_summary.columns = [
    f"{factor}_{stat if stat == 'mean' else 'std'}"
    for factor, stat in zscore_summary.columns
]
print(
    "各月因子均值绝对值最大值：",
    zscore_summary.filter(like="_mean").abs().max().max(),
)
factor_panel_zscore[["date", "stock_code", *FACTOR_COLUMNS]].head()

各月因子均值绝对值最大值： 3.851905997940436e-15


,date,stock_code,ep,bp,roe,gross_profitability,momentum_12_1,momentum_3m,low_volatility_60d,amihud_illiquidity_20d,size
0,2018-01-31,000001.SZ,2.828347,2.593782,NaN,NaN,1.810724,2.364716,-0.540494,1.192791,3.398963
1,2018-01-31,000002.SZ,1.285455,-0.438008,NaN,-0.799478,2.180934,2.791393,-1.492589,1.192791,3.398963
2,2018-01-31,000004.SZ,-0.593785,-1.472100,NaN,1.317529,-0.948762,-0.300808,-1.889396,-1.856665,-1.427017
3,2018-01-31,000005.SZ,-1.446957,-0.359342,NaN,-1.062171,-0.787832,-0.180186,0.849588,-0.791952,-0.606071
4,2018-01-31,000006.SZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. 保存结果

In [5]:
raw_path, zscore_path = save_factor_panels(
    factor_panel_raw,
    factor_panel_zscore,
    output_directory=PROCESSED_DIR,
)

print(f"原始因子面板：{raw_path}，{raw_path.stat().st_size / 1024**2:.1f} MB")
print(f"标准化因子面板：{zscore_path}，{zscore_path.stat().st_size / 1024**2:.1f} MB")

原始因子面板：/Users/mac/Desktop/a-share-multifactor/data/processed/factor_panel_raw.parquet，34.6 MB
标准化因子面板：/Users/mac/Desktop/a-share-multifactor/data/processed/factor_panel_zscore.parquet，36.3 MB


## 6. 回读校验

回读两份输出的主键和因子字段，确认写入结果完整。

In [6]:
saved_raw = pd.read_parquet(raw_path)
saved_zscore = pd.read_parquet(zscore_path)

assert tuple(saved_raw.columns) == OUTPUT_COLUMNS
assert tuple(saved_zscore.columns) == OUTPUT_COLUMNS
assert saved_raw.shape == saved_zscore.shape
assert len(saved_raw) == len(monthly_panel)
assert saved_raw[["date", "stock_code"]].equals(
    saved_zscore[["date", "stock_code"]]
)
assert not saved_raw.duplicated(["date", "stock_code"]).any()
assert not saved_zscore.duplicated(["date", "stock_code"]).any()

print(
    f"校验完成：两份精简因子面板均包含 {len(saved_raw):,} 行、"
    f"{len(FACTOR_PANEL_CONTEXT_COLUMNS)} 个上下文字段和 {len(FACTOR_COLUMNS)} 个因子。"
)

校验完成：两份精简因子面板均包含 426,474 行、13 个上下文字段和 9 个因子。
